# 01 — Download the sector-allocation market dataset

This notebook has one responsibility: download and save the daily market dataset used by the sector-allocation experiment.

- **Tradable universe:** all nine original Select Sector SPDR ETFs: XLB, XLE, XLF, XLI, XLK, XLP, XLU, XLV, and XLY
- **Context only:** SPY for the broad U.S. equity market and the VIX index for the volatility regime
- **History:** January 1999 through the latest available trading day
- **Prices:** split- and dividend-adjusted OHLC (`auto_adjust=True`)

All file paths are explicit and relative to the `notebooks/` directory, so run this notebook with `notebooks/` as the working directory.

In [ ]:
from pathlib import Path
import sys

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "portfolio_rl" / "data.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from src.portfolio_rl.data import download_market_data
from src.portfolio_rl.validation import validate_market_data


## Configuration


In [ ]:
RAW_DIR = project_root / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = "1999-01-01"
END_DATE = "2026-10-01"
MIN_TRADABLE_HISTORY_YEARS = 25

TRADABLE_SECTOR_ETFS = {
    "XLB": ("Materials", "Materials Select Sector SPDR Fund"),
    "XLE": ("Energy", "Energy Select Sector SPDR Fund"),
    "XLF": ("Financials", "Financial Select Sector SPDR Fund"),
    "XLI": ("Industrials", "Industrial Select Sector SPDR Fund"),
    "XLK": ("Information Technology", "Technology Select Sector SPDR Fund"),
    "XLP": ("Consumer Staples", "Consumer Staples Select Sector SPDR Fund"),
    "XLU": ("Utilities", "Utilities Select Sector SPDR Fund"),
    "XLV": ("Health Care", "Health Care Select Sector SPDR Fund"),
    "XLY": ("Consumer Discretionary", "Consumer Discretionary Select Sector SPDR Fund"),
}

CONTEXT_SERIES = {
    "SPY": "S&P 500 ETF",
    "^VIX": "CBOE Volatility Index",
}

print(f"Raw data directory: {RAW_DIR}")

## Build the requested symbol set

In [3]:
tradable_symbols = list(TRADABLE_SECTOR_ETFS)
context_symbols = list(CONTEXT_SERIES)
download_symbols = tradable_symbols + context_symbols

symbol_metadata = pd.DataFrame(
    [
        {
            "symbol": symbol,
            "name": label,
            "sector": sector,
            "tradable": True,
        }
        for symbol, (sector, label) in TRADABLE_SECTOR_ETFS.items()
    ]
    + [
        {
            "symbol": symbol,
            "name": label,
            "sector": pd.NA,
            "tradable": False,
        }
        for symbol, label in CONTEXT_SERIES.items()
    ]
)

print(f"Tradable assets: {len(tradable_symbols)}")
print(f"Context series:  {len(context_symbols)}")
display(symbol_metadata)

Tradable assets: 9
Context series:  2


,symbol,name,sector,tradable
0,XLB,Materials Select Sector SPDR Fund,Materials,True
1,XLE,Energy Select Sector SPDR Fund,Energy,True
2,XLF,Financial Select Sector SPDR Fund,Financials,True
3,XLI,Industrial Select Sector SPDR Fund,Industrials,True
4,XLK,Technology Select Sector SPDR Fund,Information Technology,True
5,XLP,Consumer Staples Select Sector SPDR Fund,Consumer Staples,True
6,XLU,Utilities Select Sector SPDR Fund,Utilities,True
7,XLV,Health Care Select Sector SPDR Fund,Health Care,True
8,XLY,Consumer Discretionary Select Sector SPDR Fund,Consumer Discretionary,True
9,SPY,S&P 500 ETF,<NA>,False


## Download daily data from Yahoo Finance

Yahoo's end date is exclusive when one is supplied. `END_DATE` is left unset so the notebook retrieves the latest available data.

In [ ]:
market_data = download_market_data(
    tradable_symbols, symbol_metadata, START_DATE, END_DATE,
    context_symbols=context_symbols,
)
market_data.tail()


## Normalize and validate


Normalization and validation run inside `download_market_data`.
Invalid observations raise an error before the dataset is returned.


Market rows are sorted by date and symbol. Missing-price rows are preserved
and reported by validation; they are never silently dropped.


In [ ]:
tradable_history = (
    market_data.loc[
        market_data["tradable"]
    ]
    .groupby("symbol")
    .agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        observations=("close", "count"),
    )
    .reset_index()
)

tradable_history["history_years"] = (
    tradable_history["last_date"] - tradable_history["first_date"]
).dt.days / 365.25

display(tradable_history)

assert (
    tradable_history["history_years"] >= MIN_TRADABLE_HISTORY_YEARS
).all(), "At least one sector ETF has insufficient history."

## Save the data


In [ ]:
# Recheck immediately before saving in case earlier cells changed the table.
validate_market_data(market_data, tradable_symbols, context_symbols=context_symbols)

data_path = RAW_DIR / "yahoo_daily_adjusted.csv.gz"

market_data.to_csv(
    data_path,
    index=False,
    compression="gzip",
)

print(
    f"Saved {len(market_data):,} rows for "
    f"{market_data['symbol'].nunique()} symbols."
)
print(f"Data: {data_path}")